In [3]:
import pandas as pd
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

TRAIN = ROOT / "data/raw/train"
VALIDATION = ROOT / "data/raw/validation"
TEST = ROOT / "data/raw/test"

transactions = pd.read_parquet(TRAIN / "transactions.parquet")
transfers = pd.read_parquet(TRAIN / "transfers.parquet")
blocks = pd.read_parquet(TRAIN / "blocks.parquet")
rates = pd.read_csv(TRAIN / "market_rates.csv")
labels = pd.read_csv(TRAIN / "labels.csv")

for name, df in {
    "transactions": transactions,
    "transfers": transfers,
    "blocks": blocks,
    "rates": rates,
    "labels": labels
}.items():
    print(f"\n{name}: {df.shape}")
    print("Columns:", df.columns.tolist())
    print("Missing values:", df.isna().sum().sum())


transactions: (40131, 26)
Columns: ['LEDGER_INDEX', 'TIMESTAMP', 'DATETIME', 'LEDGER_HASH', 'TRANSACTION_HASH', 'TRANSACTION_INDEX', 'ACCOUNT', 'TRANSACTION_TYPE', 'SOURCE_FEE', 'FEE', 'SEQUENCE', 'ACCOUNT_TRANSACTION_ID', 'FLAGS', 'LAST_LEDGER_SEQUENCE', 'MEMOS', 'SIGNERS', 'SOURCE_TAG', 'SIGNING_PUB_KEY', 'TICKET_SEQUENCE', 'TXN_SIGNATURE', 'TRANSACTION_RESULT', 'DELIVERED_AMOUNT', 'SUCCESS', 'EXTRA_TRANSACTION_FIELDS', 'DATE_CREATED', 'DATE_UPDATED']
Missing values: 280917

transfers: (60557, 16)
Columns: ['LEDGER_INDEX', 'TIMESTAMP', 'DATETIME', 'LEDGER_HASH', 'TRANSACTION_INDEX', 'TRANSACTION_HASH', 'TRANSFER_TYPE', 'TRANSFER_INDEX', 'FROM_ADDRESS', 'TO_ADDRESS', 'TOKEN_ADDRESS', 'SYMBOL', 'SOURCE_VALUE', 'VALUE', 'DATE_CREATED', 'DATE_UPDATED']
Missing values: 0

blocks: (151200, 17)
Columns: ['LEDGER_INDEX', 'TIMESTAMP', 'DATETIME', 'LEDGER_HASH', 'TRANSACTION_HASH', 'VALIDATED', 'ACCOUNT_HASH', 'TOTAL_COINS', 'CLOSE_FLAGS', 'CLOSE_TIME', 'CLOSE_TIME_HUMAN', 'CLOSE_TIME_RESOLUT

In [4]:
# 1. Missing values per column
print("=== TRANSACTIONS: MISSING VALUES ===")
print(
    transactions.isna().sum()
    .sort_values(ascending=False)
    .to_string()
)

print("\n=== LABELS: MISSING VALUES ===")
print(labels.isna().sum().to_string())

# 2. Duplicate checks
print("\n=== DUPLICATES ===")
print("Transactions duplicate hashes:",
      transactions["TRANSACTION_HASH"].duplicated().sum())

print("Labels duplicate hashes:",
      labels["TRANSACTION_HASH"].duplicated().sum())

print("Transfers duplicate rows:",
      transfers.duplicated().sum())

# 3. Target distribution
print("\n=== TARGET DISTRIBUTION ===")
print(labels["is_suspicious"].value_counts(dropna=False))
print(labels["is_suspicious"].value_counts(normalize=True, dropna=False))

# 4. Transaction types
print("\n=== TRANSACTION TYPES ===")
print(transactions["TRANSACTION_TYPE"].value_counts(dropna=False))

# 5. Transfer counts per transaction
print("\n=== TRANSFERS PER TRANSACTION ===")
print(transfers.groupby("TRANSACTION_HASH").size().describe())

=== TRANSACTIONS: MISSING VALUES ===
MEMOS                       40131
TXN_SIGNATURE               40131
TICKET_SEQUENCE             40131
SIGNING_PUB_KEY             40131
SOURCE_TAG                  40131
ACCOUNT_TRANSACTION_ID      40131
SIGNERS                     40131
LEDGER_INDEX                    0
DATE_CREATED                    0
EXTRA_TRANSACTION_FIELDS        0
SUCCESS                         0
DELIVERED_AMOUNT                0
TRANSACTION_RESULT              0
LAST_LEDGER_SEQUENCE            0
TIMESTAMP                       0
FLAGS                           0
SEQUENCE                        0
FEE                             0
SOURCE_FEE                      0
TRANSACTION_TYPE                0
ACCOUNT                         0
TRANSACTION_INDEX               0
TRANSACTION_HASH                0
LEDGER_HASH                     0
DATETIME                        0
DATE_UPDATED                    0

=== LABELS: MISSING VALUES ===
TRANSACTION_HASH        0
is_suspicious        

In [5]:
columns_to_drop = [
    "MEMOS",
    "TXN_SIGNATURE",
    "TICKET_SEQUENCE",
    "SIGNING_PUB_KEY",
    "SOURCE_TAG",
    "ACCOUNT_TRANSACTION_ID",
    "SIGNERS"
]

transactions_clean = transactions.drop(
    columns=columns_to_drop
).copy()

print("Before:", transactions.shape)
print("After:", transactions_clean.shape)
print("Missing values:", transactions_clean.isna().sum().sum())

Before: (40131, 26)
After: (40131, 19)
Missing values: 0


We check for duplicate transaction hashes and missing target labels to ensure data consistency. 

In [6]:
print("=== DUPLICATES ===")

print("Duplicate transaction hashes:",
      transactions_clean["TRANSACTION_HASH"].duplicated().sum())

print("Duplicate label hashes:",
      labels["TRANSACTION_HASH"].duplicated().sum())

print("\n=== TARGET ===")

print("Missing target values:",
      labels["is_suspicious"].isna().sum())

print("\nTarget distribution:")
print(labels["is_suspicious"].value_counts())

print("\nTarget percentages:")
print(labels["is_suspicious"].value_counts(normalize=True) * 100)

=== DUPLICATES ===
Duplicate transaction hashes: 0
Duplicate label hashes: 0

=== TARGET ===
Missing target values: 0

Target distribution:
is_suspicious
0    38881
1     1250
Name: count, dtype: int64

Target percentages:
is_suspicious
0    96.885201
1     3.114799
Name: proportion, dtype: float64


Some transactions contain multiple transfers. We aggregate these records by transaction hash to create four features: the number of transfers, unique senders, unique receivers, and unique tokens. This gives us additional information about each transaction without creating duplicate rows.

In [7]:
transfer_features = transfers.groupby("TRANSACTION_HASH").agg(
    transfer_count=("TRANSFER_INDEX", "count"),
    unique_senders=("FROM_ADDRESS", "nunique"),
    unique_receivers=("TO_ADDRESS", "nunique"),
    unique_tokens=("SYMBOL", "nunique")
).reset_index()

print("Shape:", transfer_features.shape)
display(transfer_features.head())

Shape: (40131, 5)


,TRANSACTION_HASH,transfer_count,unique_senders,unique_receivers,unique_tokens
0,00017BB7FD900622F10386CAF5F82BFD5575881005E331...,1,1,1,1
1,00028A6E2260BEDF874174AD4DBC96B139F85A4E9ABD9D...,3,3,3,3
2,00032386B56944BF3295207787EAF0A7C3C85947494B66...,1,1,1,1
3,000995ECC9CC8783B4C0612EA4AEE4A428B848DA7952C1...,3,3,3,3
4,0009B43313655466F274B66204B71902F616989ECC9138...,3,3,3,3


We merge the aggregated transfer features with the cleaned transactions using TRANSACTION_HASH. A left join ensures that all original transactions are preserved. For transactions without matching transfers, the new count features are set to zero.

In [8]:
# Merge transaction data with aggregated transfer features
train_features = transactions_clean.merge(
    transfer_features,
    on="TRANSACTION_HASH",
    how="left",
    validate="one_to_one"
)

# Transactions without transfer records receive zero counts
feature_cols = [
    "transfer_count",
    "unique_senders",
    "unique_receivers",
    "unique_tokens"
]

train_features[feature_cols] = (
    train_features[feature_cols].fillna(0).astype(int)
)

print("Original transactions:", len(transactions_clean))
print("After merge:", len(train_features))
print("Duplicate hashes:",
      train_features["TRANSACTION_HASH"].duplicated().sum())

print("\nMissing values:")
print(train_features.isna().sum().sum())

print("\nNew shape:", train_features.shape)

display(train_features[
    ["TRANSACTION_HASH"] + feature_cols
].head())

Original transactions: 40131
After merge: 40131
Duplicate hashes: 0

Missing values:
0

New shape: (40131, 23)


,TRANSACTION_HASH,transfer_count,unique_senders,unique_receivers,unique_tokens
0,954736A6E1BB5A3433EB916D8221F2DF5938A521EA2222...,1,1,1,1
1,C93CF387C2148608329ED89420105EFD883020CDBFDFAE...,1,1,1,1
2,E46BFC5EC409B486E1C97F20BCB62AE96432B20937D265...,1,1,1,1
3,E7CAB61BBDAC502147EF393C7EA93A2A937DA3F01623E2...,3,3,3,3
4,E9549020D2717DB401CA4C30BB82500758ABF100600D9C...,2,2,2,2


We add is_suspicious to the training dataset, indicating whether each transaction is suspicious (1) or normal (0). We exclude other label-related metadata to avoid introducing target leakage into the model.

In [9]:
train_features = train_features.merge(
    labels[["TRANSACTION_HASH", "is_suspicious"]],
    on="TRANSACTION_HASH",
    how="left",
    validate="one_to_one"
)

print("Shape:", train_features.shape)
print("Missing labels:", train_features["is_suspicious"].isna().sum())

Shape: (40131, 24)
Missing labels: 0


Preparing Features for Machine Learning

We inspect the data types and sample values to identify which columns can be used directly for training and which require further transformation or encoding.

In [10]:
print("=== COLUMN TYPES ===")
print(train_features.dtypes.to_string())

print("\n=== SAMPLE DATA ===")
display(train_features.head(3))

=== COLUMN TYPES ===
LEDGER_INDEX                             object
TIMESTAMP                                object
DATETIME                    datetime64[ms, UTC]
LEDGER_HASH                                 str
TRANSACTION_HASH                            str
TRANSACTION_INDEX                        object
ACCOUNT                                     str
TRANSACTION_TYPE                            str
SOURCE_FEE                               object
FEE                                     float64
SEQUENCE                                 object
FLAGS                                    object
LAST_LEDGER_SEQUENCE                     object
TRANSACTION_RESULT                          str
DELIVERED_AMOUNT                            str
SUCCESS                                    bool
EXTRA_TRANSACTION_FIELDS                    str
DATE_CREATED                datetime64[ms, UTC]
DATE_UPDATED                datetime64[ms, UTC]
transfer_count                            int64
unique_senders     

,LEDGER_INDEX,TIMESTAMP,DATETIME,LEDGER_HASH,TRANSACTION_HASH,TRANSACTION_INDEX,ACCOUNT,TRANSACTION_TYPE,SOURCE_FEE,FEE,...,DELIVERED_AMOUNT,SUCCESS,EXTRA_TRANSACTION_FIELDS,DATE_CREATED,DATE_UPDATED,transfer_count,unique_senders,unique_receivers,unique_tokens,is_suspicious
0,1124529,1788718916,2026-09-06 18:21:56+00:00,6F7094E1DDA27405855AB2CE4E97A555E0B372D45625B3...,954736A6E1BB5A3433EB916D8221F2DF5938A521EA2222...,0,r3CWFt1SXiykW92LwKU19CvfTFcFRnbnDY,Payment,10,0.000010,...,"""1167184470""",True,"{""Amount"":""1167184470"",""DeliverMax"":""116718447...",2026-09-06 18:21:56+00:00,2026-09-06 18:21:56+00:00,1,1,1,1,0
1,1136375,1788766300,2026-09-07 07:31:40+00:00,6760D562E18F762A5068E23BAC9CDE21AE43D11C44E466...,C93CF387C2148608329ED89420105EFD883020CDBFDFAE...,1,rEFG2WG1W8bHQhHp2P1rcxRBMxmz9C52B7,Payment,15,0.000015,...,"""486435229""",True,"{""Amount"":""486435229"",""DeliverMax"":""486435229""...",2026-09-07 07:31:40+00:00,2026-09-07 07:31:40+00:00,1,1,1,1,0
2,1129067,1788737068,2026-09-06 23:24:28+00:00,FCBE03345724C312C98168B9F2C4C0FA2B2A181B6FBFA3...,E46BFC5EC409B486E1C97F20BCB62AE96432B20937D265...,1,rJxqpGRZ2bJtB1Q8HYCV9iPon9HpvJugZm,Payment,15,0.000015,...,"{""currency"": ""EUR"", ""issuer"": ""rLeMq5iFyEhgHLf...",True,"{""Amount"":{""currency"":""EUR"",""issuer"":""rLeMq5iF...",2026-09-06 23:24:28+00:00,2026-09-06 23:24:28+00:00,1,1,1,1,0


In [11]:
print("=== DATA TYPES ===")
print(train_features.dtypes.to_string())

print("\n=== NON-NUMERIC COLUMNS ===")
print(
    train_features.select_dtypes(
        exclude=["number", "bool"]
    ).columns.tolist()
)

print("\n=== SAMPLE VALUES ===")
display(train_features.head(3))

=== DATA TYPES ===
LEDGER_INDEX                             object
TIMESTAMP                                object
DATETIME                    datetime64[ms, UTC]
LEDGER_HASH                                 str
TRANSACTION_HASH                            str
TRANSACTION_INDEX                        object
ACCOUNT                                     str
TRANSACTION_TYPE                            str
SOURCE_FEE                               object
FEE                                     float64
SEQUENCE                                 object
FLAGS                                    object
LAST_LEDGER_SEQUENCE                     object
TRANSACTION_RESULT                          str
DELIVERED_AMOUNT                            str
SUCCESS                                    bool
EXTRA_TRANSACTION_FIELDS                    str
DATE_CREATED                datetime64[ms, UTC]
DATE_UPDATED                datetime64[ms, UTC]
transfer_count                            int64
unique_senders       

,LEDGER_INDEX,TIMESTAMP,DATETIME,LEDGER_HASH,TRANSACTION_HASH,TRANSACTION_INDEX,ACCOUNT,TRANSACTION_TYPE,SOURCE_FEE,FEE,...,DELIVERED_AMOUNT,SUCCESS,EXTRA_TRANSACTION_FIELDS,DATE_CREATED,DATE_UPDATED,transfer_count,unique_senders,unique_receivers,unique_tokens,is_suspicious
0,1124529,1788718916,2026-09-06 18:21:56+00:00,6F7094E1DDA27405855AB2CE4E97A555E0B372D45625B3...,954736A6E1BB5A3433EB916D8221F2DF5938A521EA2222...,0,r3CWFt1SXiykW92LwKU19CvfTFcFRnbnDY,Payment,10,0.000010,...,"""1167184470""",True,"{""Amount"":""1167184470"",""DeliverMax"":""116718447...",2026-09-06 18:21:56+00:00,2026-09-06 18:21:56+00:00,1,1,1,1,0
1,1136375,1788766300,2026-09-07 07:31:40+00:00,6760D562E18F762A5068E23BAC9CDE21AE43D11C44E466...,C93CF387C2148608329ED89420105EFD883020CDBFDFAE...,1,rEFG2WG1W8bHQhHp2P1rcxRBMxmz9C52B7,Payment,15,0.000015,...,"""486435229""",True,"{""Amount"":""486435229"",""DeliverMax"":""486435229""...",2026-09-07 07:31:40+00:00,2026-09-07 07:31:40+00:00,1,1,1,1,0
2,1129067,1788737068,2026-09-06 23:24:28+00:00,FCBE03345724C312C98168B9F2C4C0FA2B2A181B6FBFA3...,E46BFC5EC409B486E1C97F20BCB62AE96432B20937D265...,1,rJxqpGRZ2bJtB1Q8HYCV9iPon9HpvJugZm,Payment,15,0.000015,...,"{""currency"": ""EUR"", ""issuer"": ""rLeMq5iFyEhgHLf...",True,"{""Amount"":{""currency"":""EUR"",""issuer"":""rLeMq5iF...",2026-09-06 23:24:28+00:00,2026-09-06 23:24:28+00:00,1,1,1,1,0


Some columns are stored as objects even though they represent numerical values. We convert these columns to numeric types so they can be used by machine learning models, while checking for any values that cannot be converted.

We keep identifiers, categorical variables, and complex fields separate for further processing.

In [12]:
# Convert numerical columns stored as objects
numeric_columns = [
    "LEDGER_INDEX",
    "TIMESTAMP",
    "TRANSACTION_INDEX",
    "SOURCE_FEE",
    "SEQUENCE",
    "FLAGS",
    "LAST_LEDGER_SEQUENCE"
]

train_features_numeric = train_features.copy()

for col in numeric_columns:
    train_features_numeric[col] = pd.to_numeric(
        train_features_numeric[col],
        errors="coerce"
    )

print("=== CONVERTED DATA TYPES ===")
print(train_features_numeric[numeric_columns].dtypes)

print("\n=== VALUES THAT COULD NOT BE CONVERTED ===")
print(train_features_numeric[numeric_columns].isna().sum())

=== CONVERTED DATA TYPES ===
LEDGER_INDEX            float64
TIMESTAMP               float64
TRANSACTION_INDEX       float64
SOURCE_FEE              float64
SEQUENCE                float64
FLAGS                   float64
LAST_LEDGER_SEQUENCE    float64
dtype: object

=== VALUES THAT COULD NOT BE CONVERTED ===
LEDGER_INDEX            0
TIMESTAMP               0
TRANSACTION_INDEX       0
SOURCE_FEE              0
SEQUENCE                0
FLAGS                   0
LAST_LEDGER_SEQUENCE    0
dtype: int64


Machine learning models generally require numerical inputs, so we convert categorical variables into numerical features using one-hot encoding.

In [13]:
# Identify categories before encoding
categorical_columns = [
    "TRANSACTION_TYPE",
    "TRANSACTION_RESULT"
]

for col in categorical_columns:
    print(f"\n=== {col} ===")
    print(train_features_numeric[col].value_counts(dropna=False))

# Apply one-hot encoding
train_features_encoded = pd.get_dummies(
    train_features_numeric,
    columns=categorical_columns,
    dtype=int
)

print("\nOriginal shape:", train_features_numeric.shape)
print("Encoded shape:", train_features_encoded.shape)

print("\nNew categorical features:")
print([
    col for col in train_features_encoded.columns
    if col.startswith(("TRANSACTION_TYPE_", "TRANSACTION_RESULT_"))
])


=== TRANSACTION_TYPE ===
TRANSACTION_TYPE
Payment    40131
Name: count, dtype: int64

=== TRANSACTION_RESULT ===
TRANSACTION_RESULT
tesSUCCESS    40131
Name: count, dtype: int64

Original shape: (40131, 24)
Encoded shape: (40131, 24)

New categorical features:
['TRANSACTION_TYPE_Payment', 'TRANSACTION_RESULT_tesSUCCESS']


Both TRANSACTION_TYPE and TRANSACTION_RESULT contain only one category across the training dataset (Payment and tesSUCCESS, respectively).

Since these features have no variation, they cannot help the model distinguish suspicious from normal transactions. We remove their encoded columns to reduce unnecessary features.

In [14]:
# Remove encoded columns that contain only one unique value
constant_columns = [
    "TRANSACTION_TYPE_Payment",
    "TRANSACTION_RESULT_tesSUCCESS"
]

train_features_encoded = train_features_encoded.drop(
    columns=constant_columns
)

print("Removed columns:", constant_columns)
print("New shape:", train_features_encoded.shape)

Removed columns: ['TRANSACTION_TYPE_Payment', 'TRANSACTION_RESULT_tesSUCCESS']
New shape: (40131, 22)


Transaction timing may help identify suspicious activity. We extract the hour of the day, day of the week, and whether a transaction occurred during the weekend.

These features allow our models to capture temporal patterns without using raw datetime values directly.

In [15]:
# Extract temporal features
train_features_encoded["hour"] = (
    train_features_encoded["DATETIME"].dt.hour
)

train_features_encoded["day_of_week"] = (
    train_features_encoded["DATETIME"].dt.dayofweek
)

train_features_encoded["is_weekend"] = (
    (train_features_encoded["day_of_week"] >= 5).astype(int)
)

print("New shape:", train_features_encoded.shape)

display(
    train_features_encoded[
        ["DATETIME", "hour", "day_of_week", "is_weekend"]
    ].head()
)

New shape: (40131, 25)


/var/folders/n6/z6d7zfwx0rx98wcl7526tg880000gn/T/ipykernel_81778/3013767911.py:7: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  train_features_encoded["DATETIME"].dt.dayofweek


,DATETIME,hour,day_of_week,is_weekend
0,2026-09-06 18:21:56+00:00,18,6,1
1,2026-09-07 07:31:40+00:00,7,0,0
2,2026-09-06 23:24:28+00:00,23,6,1
3,2026-09-05 02:20:20+00:00,2,5,1
4,2026-09-06 15:19:28+00:00,15,6,1


Some transaction attributes are stored as text or structured data rather than simple numerical values. We inspect these fields to determine whether they contain useful information that can be extracted for machine learning.

We avoid dropping them prematurely, as transaction amounts and additional attributes may help identify suspicious activity.

In [16]:
complex_columns = [
    "DELIVERED_AMOUNT",
    "EXTRA_TRANSACTION_FIELDS"
]

for col in complex_columns:
    print(f"\n=== {col} ===")
    print("Data type:", train_features_encoded[col].dtype)
    print("Unique values:", train_features_encoded[col].nunique())

    print("\nSample values:")
    for value in train_features_encoded[col].head(5):
        print(repr(value))


=== DELIVERED_AMOUNT ===
Data type: str
Unique values: 40131

Sample values:
'"1167184470"'
'"486435229"'
'{"currency": "EUR", "issuer": "rLeMq5iFyEhgHLfBxeu4eRQWEGGdeL8hXK", "value": "1.715004"}'
'{"currency": "USD", "issuer": "rLeMq5iFyEhgHLfBxeu4eRQWEGGdeL8hXK", "value": "12.323166"}'
'"163125642"'

=== EXTRA_TRANSACTION_FIELDS ===
Data type: str
Unique values: 40131

Sample values:
'{"Amount":"1167184470","DeliverMax":"1167184470","Destination":"rMWFzNYApzeKVcHyXGSUz2XQpuHmSA11kU"}'
'{"Amount":"486435229","DeliverMax":"486435229","Destination":"rDwLP5NUvC25gwBDHgv5qP9nCZjT8kuZSy"}'
'{"Amount":{"currency":"EUR","issuer":"rLeMq5iFyEhgHLfBxeu4eRQWEGGdeL8hXK","value":"1.715004"},"DeliverMax":{"currency":"EUR","issuer":"rLeMq5iFyEhgHLfBxeu4eRQWEGGdeL8hXK","value":"1.715004"},"Destination":"rsSAFra1UVoobuNrQzh7HJHepjvSTkHyAM"}'
'{"Amount":{"currency":"USD","issuer":"rLeMq5iFyEhgHLfBxeu4eRQWEGGdeL8hXK","value":"12.323166"},"DeliverMax":{"currency":"USD","issuer":"rLeMq5iFyEhgHLfBxeu4eRQW

Payment amounts are stored in different formats depending on the currency. XRP payments use drops, while issued tokens store their currency and value in a JSON object.

We parse these fields to extract numerical amounts, currencies, and additional payment characteristics. Currency conversion will be handled separately to ensure amounts are comparable across different assets.

In [17]:
import json
import pandas as pd

def parse_delivered_amount(value):
    amount = json.loads(value)

    if isinstance(amount, dict):
        return pd.Series({
            "delivered_value": float(amount["value"]),
            "delivered_currency": amount["currency"],
            "is_xrp_payment": 0
        })

    return pd.Series({
        "delivered_value": float(amount) / 1_000_000,
        "delivered_currency": "XRP",
        "is_xrp_payment": 1
    })


# Parse delivered amounts
amount_features = train_features_encoded[
    "DELIVERED_AMOUNT"
].apply(parse_delivered_amount)

train_features_encoded = pd.concat(
    [train_features_encoded, amount_features],
    axis=1
)

# Parse additional transaction fields
extra_fields = train_features_encoded[
    "EXTRA_TRANSACTION_FIELDS"
].apply(json.loads)

train_features_encoded["has_sendmax"] = extra_fields.apply(
    lambda x: int("SendMax" in x)
)

train_features_encoded["has_paths"] = extra_fields.apply(
    lambda x: int(bool(x.get("Paths")))
)

display(
    train_features_encoded[
        [
            "delivered_value",
            "delivered_currency",
            "is_xrp_payment",
            "has_sendmax",
            "has_paths"
        ]
    ].head()
)

,delivered_value,delivered_currency,is_xrp_payment,has_sendmax,has_paths
0,1167.184470,XRP,1,0,0
1,486.435229,XRP,1,0,0
2,1.715004,EUR,0,0,0
3,12.323166,USD,0,1,1
4,163.125642,XRP,1,1,0


Transaction amounts are expressed in different currencies, so their raw values are not directly comparable. We inspect the market exchange rates to determine how to convert payment amounts into USD.

This will allow us to create a consistent numerical feature for transaction value across different currencies.

In [18]:
print("=== MARKET RATES ===")
print("Shape:", rates.shape)
print("Data types:")
print(rates.dtypes)

print("\nAvailable assets:")
print(rates["ASSET"].value_counts())

print("\nSample rates:")
display(rates.head(10))

print("\nTransaction currencies:")
print(
    train_features_encoded["delivered_currency"]
    .value_counts()
)

=== MARKET RATES ===
Shape: (850, 3)
Data types:
TIMESTAMP          int64
ASSET                str
PRICE_MICRO_USD    int64
dtype: object

Available assets:
ASSET
XRP                                       170
r99vy4o5fXhC3PwEXEpnc1f9Mss4aCzYBW:USD    170
r99vy4o5fXhC3PwEXEpnc1f9Mss4aCzYBW:GBP    170
rLeMq5iFyEhgHLfBxeu4eRQWEGGdeL8hXK:USD    170
rLeMq5iFyEhgHLfBxeu4eRQWEGGdeL8hXK:EUR    170
Name: count, dtype: int64

Sample rates:


,TIMESTAMP,ASSET,PRICE_MICRO_USD
0,1788220800,XRP,651148
1,1788220800,r99vy4o5fXhC3PwEXEpnc1f9Mss4aCzYBW:USD,1001924
2,1788220800,r99vy4o5fXhC3PwEXEpnc1f9Mss4aCzYBW:GBP,1297809
3,1788220800,rLeMq5iFyEhgHLfBxeu4eRQWEGGdeL8hXK:USD,999324
4,1788220800,rLeMq5iFyEhgHLfBxeu4eRQWEGGdeL8hXK:EUR,1099502
5,1788224400,XRP,650730
6,1788224400,r99vy4o5fXhC3PwEXEpnc1f9Mss4aCzYBW:USD,1000931
7,1788224400,r99vy4o5fXhC3PwEXEpnc1f9Mss4aCzYBW:GBP,1296430
8,1788224400,rLeMq5iFyEhgHLfBxeu4eRQWEGGdeL8hXK:USD,1000268
9,1788224400,rLeMq5iFyEhgHLfBxeu4eRQWEGGdeL8hXK:EUR,1097541



Transaction currencies:
delivered_currency
XRP    12921
USD    12788
GBP     7942
EUR     6480
Name: count, dtype: int64


To make transaction amounts comparable across currencies, we convert all delivered values into USD using the historical market rates.

We identify each asset by its currency and issuer, then match each transaction with the most recent available exchange rate at or before its timestamp. This avoids using future exchange-rate information.

In [21]:
import json
import pandas as pd

# Identify each asset using currency and issuer
def get_asset(raw_amount):
    amount = json.loads(raw_amount)

    if isinstance(amount, dict):
        return f"{amount['issuer']}:{amount['currency']}"

    return "XRP"


usd_features = train_features_encoded.copy()

usd_features["asset"] = usd_features[
    "DELIVERED_AMOUNT"
].apply(get_asset)

# Prepare exchange rates
rates_usd = rates.copy()
rates_usd["usd_price"] = (
    rates_usd["PRICE_MICRO_USD"] / 1_000_000
)

# Ensure both timestamp columns use the same numeric type
rates_usd["TIMESTAMP"] = pd.to_numeric(
    rates_usd["TIMESTAMP"]
).astype("float64")

usd_features["TIMESTAMP"] = pd.to_numeric(
    usd_features["TIMESTAMP"]
).astype("float64")

# Match each transaction to the latest known rate
usd_features = pd.merge_asof(
    usd_features.sort_values("TIMESTAMP"),
    rates_usd[["TIMESTAMP", "ASSET", "usd_price"]]
        .sort_values("TIMESTAMP"),
    left_on="TIMESTAMP",
    right_on="TIMESTAMP",
    left_by="asset",
    right_by="ASSET",
    direction="backward"
)

# Convert to USD
usd_features["delivered_usd"] = (
    usd_features["delivered_value"] *
    usd_features["usd_price"]
)

print("Transactions:", len(usd_features))
print("Missing exchange rates:",
      usd_features["usd_price"].isna().sum())

display(
    usd_features[
        ["delivered_currency", "delivered_value",
         "usd_price", "delivered_usd"]
    ].head()
)

Transactions: 40131
Missing exchange rates: 0


,delivered_currency,delivered_value,usd_price,delivered_usd
0,USD,25.613174,1.001924,25.662454
1,USD,6.412834,0.999324,6.408499
2,USD,11.323993,0.999324,11.316338
3,USD,172.268685,0.999324,172.152231
4,USD,11.327982,1.001924,11.349777


We verify that all transactions have been matched with an exchange rate and successfully converted into USD. We also check for missing values, duplicate transactions, and invalid amounts to ensure the resulting feature is reliable for machine learning.

In [22]:
print("=== USD CONVERSION VALIDATION ===")

print("Total transactions:", len(usd_features))

print("Missing exchange rates:",
      usd_features["usd_price"].isna().sum())

print("Missing USD amounts:",
      usd_features["delivered_usd"].isna().sum())

print("Duplicate transactions:",
      usd_features["TRANSACTION_HASH"].duplicated().sum())

print("Negative USD amounts:",
      (usd_features["delivered_usd"] < 0).sum())

print("\n=== USD AMOUNT STATISTICS ===")
print(usd_features["delivered_usd"].describe())

print("\n=== MISSING RATES BY CURRENCY ===")
print(
    usd_features.groupby("delivered_currency")["usd_price"]
    .apply(lambda x: x.isna().sum())
)

=== USD CONVERSION VALIDATION ===
Total transactions: 40131
Missing exchange rates: 0
Missing USD amounts: 0
Duplicate transactions: 0
Negative USD amounts: 0

=== USD AMOUNT STATISTICS ===
count    40131.000000
mean       141.469390
std        232.712273
min          0.474892
25%         22.664823
50%         59.444801
75%        159.444799
max       4575.674819
Name: delivered_usd, dtype: float64

=== MISSING RATES BY CURRENCY ===
delivered_currency
EUR    0
GBP    0
USD    0
XRP    0
Name: usd_price, dtype: int64


In [23]:
# Export cleaned transactions for graph and behavioural features
OUTPUT_DIR = ROOT / "data/processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train_features.to_parquet(
    OUTPUT_DIR / "train_cleaned.parquet",
    index=False
)

print("Saved cleaned transactions:", len(train_features))

Saved cleaned transactions: 40131


In [ ]:
from pathlib import Path
import pandas as pd
import json

OUTPUT_DIR = ROOT / "data/processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DROP_COLS = [
    "MEMOS", "TXN_SIGNATURE", "TICKET_SEQUENCE",
    "SIGNING_PUB_KEY", "SOURCE_TAG",
    "ACCOUNT_TRANSACTION_ID", "SIGNERS"
]

def process_dataset(folder, include_labels=False):
    tx = pd.read_parquet(folder / "transactions.parquet")
    tr = pd.read_parquet(folder / "transfers.parquet")
    exchange = pd.read_csv(folder / "market_rates.csv")

    # Remove empty fields
    tx = tx.drop(columns=DROP_COLS, errors="ignore")

    # Aggregate transfers by transaction
    agg = tr.groupby("TRANSACTION_HASH").agg(
        transfer_count=("TRANSFER_INDEX", "count"),
        unique_senders=("FROM_ADDRESS", "nunique"),
        unique_receivers=("TO_ADDRESS", "nunique"),
        unique_tokens=("SYMBOL", "nunique")
    ).reset_index()

    tx = tx.merge(
        agg, on="TRANSACTION_HASH",
        how="left", validate="one_to_one"
    )

    count_cols = [
        "transfer_count", "unique_senders",
        "unique_receivers", "unique_tokens"
    ]
    tx[count_cols] = tx[count_cols].fillna(0).astype(int)

    # Temporal features
    dt = pd.to_datetime(tx["DATETIME"], utc=True)
    tx["hour"] = dt.dt.hour
    tx["day_of_week"] = dt.dt.day_of_week
    x["is_weekend"] = (dt.dt.day_of_week >= 5).astype(int)

    # Extract payment information
    def parse_amount(raw):
        value = json.loads(raw)
        if isinstance(value, dict):
            return (
                float(value["value"]),
                value["currency"],
                f'{value["issuer"]}:{value["currency"]}'
            )
        return float(value) / 1_000_000, "XRP", "XRP"

    parsed = tx["DELIVERED_AMOUNT"].map(parse_amount)

    tx["delivered_value"] = parsed.map(lambda x: x[0])
    tx["delivered_currency"] = parsed.map(lambda x: x[1])
    tx["asset"] = parsed.map(lambda x: x[2])
    tx["is_xrp_payment"] = (
        tx["delivered_currency"] == "XRP"
    ).astype(int)

    extra = tx["EXTRA_TRANSACTION_FIELDS"].map(json.loads)
    tx["has_sendmax"] = extra.map(
        lambda x: int("SendMax" in x)
    )
    tx["has_paths"] = extra.map(
        lambda x: int(bool(x.get("Paths")))
    )

    # Historical USD conversion
    tx["TIMESTAMP"] = pd.to_numeric(
        tx["TIMESTAMP"]
    ).astype("float64")

    exchange["TIMESTAMP"] = exchange[
        "TIMESTAMP"
    ].astype("float64")

    exchange["usd_price"] = (
        exchange["PRICE_MICRO_USD"] / 1_000_000
    )

    tx = pd.merge_asof(
        tx.sort_values("TIMESTAMP"),
        exchange[["TIMESTAMP", "ASSET", "usd_price"]]
            .sort_values("TIMESTAMP"),
        on="TIMESTAMP",
        left_by="asset",
        right_by="ASSET",
        direction="backward"
    )

    tx["delivered_usd"] = (
        tx["delivered_value"] * tx["usd_price"]
    )

    # Add target only for training
    if include_labels:
        lbl = pd.read_csv(
            folder / "labels.csv",
            usecols=["TRANSACTION_HASH", "is_suspicious"]
        )
        tx = tx.merge(
            lbl, on="TRANSACTION_HASH",
            how="left", validate="one_to_one"
        )
        assert tx["is_suspicious"].notna().all()

    assert len(tx) == pd.read_parquet(
        folder / "transactions.parquet"
    ).shape[0]
    assert not tx["TRANSACTION_HASH"].duplicated().any()

    return tx, tr


for split, folder in [
    ("train", TRAIN),
    ("validation", VALIDATION),
    ("test", TEST)
]:
    clean_tx, clean_transfers = process_dataset(
        folder, include_labels=(split == "train")
    )

    clean_tx.to_parquet(
        OUTPUT_DIR / f"{split}_cleaned.parquet",
        index=False
    )

    clean_transfers.to_parquet(
        OUTPUT_DIR / f"{split}_transfers.parquet",
        index=False
    )

    print(
        f"{split}: {len(clean_tx):,} transactions, "
        f"{len(clean_transfers):,} transfers, "
        f"missing USD values: "
        f"{clean_tx['delivered_usd'].isna().sum()}"
    )

print("\nAll files saved to:", OUTPUT_DIR)

/var/folders/n6/z6d7zfwx0rx98wcl7526tg880000gn/T/ipykernel_81778/1002314632.py:44: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  tx["day_of_week"] = dt.dt.dayofweek
/var/folders/n6/z6d7zfwx0rx98wcl7526tg880000gn/T/ipykernel_81778/1002314632.py:45: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  tx["is_weekend"] = (dt.dt.dayofweek >= 5).astype(int)


train: 40,131 transactions, 60,557 transfers, missing USD values: 0


/var/folders/n6/z6d7zfwx0rx98wcl7526tg880000gn/T/ipykernel_81778/1002314632.py:44: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  tx["day_of_week"] = dt.dt.dayofweek
/var/folders/n6/z6d7zfwx0rx98wcl7526tg880000gn/T/ipykernel_81778/1002314632.py:45: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  tx["is_weekend"] = (dt.dt.dayofweek >= 5).astype(int)


validation: 33,862 transactions, 55,897 transfers, missing USD values: 0


/var/folders/n6/z6d7zfwx0rx98wcl7526tg880000gn/T/ipykernel_81778/1002314632.py:44: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  tx["day_of_week"] = dt.dt.dayofweek
/var/folders/n6/z6d7zfwx0rx98wcl7526tg880000gn/T/ipykernel_81778/1002314632.py:45: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  tx["is_weekend"] = (dt.dt.dayofweek >= 5).astype(int)


test: 40,101 transactions, 65,272 transfers, missing USD values: 0

All files saved to: /Users/marinescucatinca/Hackaton/decode-detect-hackathon/data/processed
